# AlexNet giản lược · Chihuahua vs Muffin · Kaggle

Bản làm lại từ đầu, theo đúng khung repo ResNet-18 của nhóm: `data_process.py` → `alexnet_scratch.py` → `train.py`.
**Không augmentation, không EMA, không label smoothing**: ảnh chỉ được cắt giữa cho vừa đầu vào 224×224, chuẩn hóa, rồi cho mô hình học.

| | Thiết lập |
|---|---|
| Dữ liệu | thư mục `Processed/` (ảnh RGB, cạnh ngắn 256) + `stats.json`; train 4.260 · val 473 (10% của train/, seed 10) · test 1.184 |
| Tiền xử lý | CenterCrop 224 → ToTensor → Normalize (mean/std của train), giống nhau cho train, val, test |
| Mô hình | AlexNet 5 conv (11-5-3-3-3) + BatchNorm, Global Average Pooling, FC 256-512-256-2, Dropout 0,5 · 2.734.274 tham số |
| Huấn luyện | CrossEntropy, Adam (lr 1e-3, weight decay 1e-4), cosine 30 epoch, batch 32, seed 42 |
| Chọn mô hình | lưu trọng số có **val accuracy** cao nhất; test chỉ đánh giá một lần với trọng số đó |

**Cách chạy trên Kaggle**
1. *Add Input* → dataset ảnh đã dùng cho v1/v2 (upload từ `kaggle_dataset_alexnet.zip`, có `Processed/stats.json`).
2. *Accelerator* = GPU (T4 hoặc P100; code chỉ dùng 1 GPU). Internet không bắt buộc.
3. *Run All*. Nếu log báo "bỏ qua xuất ONNX" mà nhóm cần file `.onnx`: bật Internet, chạy `!pip install onnxscript`, rồi chạy lại.

**Kết quả trong `/kaggle/working`:** `alexnet_scratch_best.pth`, `alexnet_scratch_history.csv`, `alexnet_scratch.onnx` + `alexnet_scratch.onnx.data` (hai file phải đi cùng nhau),
`alexnet_scratch_curves.png`, `alexnet_scratch_cm.png`, `alexnet_scratch_test_predictions.csv`.

| Cell | Nội dung |
|---|---|
| 1–3 | Ghi `data_process.py`, `alexnet_scratch.py`, `train.py` (y hệt các file trong thư mục `alexnet_simple/`) |
| 4 | Huấn luyện 30 epoch, in loss/accuracy từng epoch và test accuracy |
| 5 | Đường cong loss / accuracy |
| 6 | Báo cáo trên test (precision, recall, F1, ROC-AUC, confusion matrix) + dự đoán từng ảnh |

In [ ]:
%%writefile data_process.py
"""Dữ liệu cho AlexNet: dùng thẳng thư mục Processed của repo ResNet-18 (ảnh RGB, cạnh ngắn 256) và stats.json.

mean/std trong stats.json được repo ResNet-18 tính trên phần train (không gồm val), sau CenterCrop 224.
Cách chia val giống hệt repo đó, nên AlexNet và ResNet-18 có cùng train/val/test.
"""
from pathlib import Path

import torch

# Kaggle: dataset được gắn vào /kaggle/input; local: thư mục "processed data" nằm cạnh thư mục này,
# hoặc Data/Processed khi thư mục này nằm trong repo ResNet-18
SEARCH_FOLDERS = [Path("/kaggle/input"), Path(__file__).resolve().parent.parent / "processed data",
                  Path(__file__).resolve().parent.parent / "Data"]
found = [path for folder in SEARCH_FOLDERS if folder.exists() for path in folder.rglob("stats.json")]
assert found, "Không tìm thấy Processed/stats.json. Trên Kaggle: Add Input → dataset ảnh đã xử lý"

STATS_FILE = str(found[0])
PROCESSED_FOLDER = str(Path(STATS_FILE).parent)

SEED = 10
VAL_RATIO = 0.1
IMAGE_SIZE = 224


def split_train_val(number_of_images):
    generator = torch.Generator().manual_seed(SEED)
    indices = torch.randperm(number_of_images, generator=generator).tolist()
    number_val = int(VAL_RATIO * number_of_images)
    train_indices = indices[number_val:]
    val_indices = indices[:number_val]
    return train_indices, val_indices


In [ ]:
%%writefile alexnet_scratch.py
"""AlexNet huấn luyện từ đầu cho ảnh 3×224×224, 2 lớp (viết theo kiểu resnet18_scratch.py).

Giữ 5 lớp conv của AlexNet: kernel 11-5-3-3-3, số kênh 64-192-384-256-256, MaxPool 3×3 bước 2.
Hai thay đổi để học được trên 4.260 ảnh:
  - BatchNorm sau mỗi conv thay cho LRN;
  - Global Average Pooling + FC 256 → 512 → 256 → 2 thay cho khối FC 9216 → 4096 → 4096,
    giảm từ 61,1 triệu xuống 2,73 triệu tham số.
"""
import torch
from torch import nn


def conv_block(in_channels, out_channels, kernel_size, stride=1, padding=0, pool=False):
    layers = [
        nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False),  # BN đã có bias
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    ]
    if pool:
        layers.append(nn.MaxPool2d(kernel_size=3, stride=2))
    return nn.Sequential(*layers)


class AlexNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 64, kernel_size=11, stride=4, padding=2, pool=True),   # 64 × 27 × 27
            conv_block(64, 192, kernel_size=5, padding=2, pool=True),            # 192 × 13 × 13
            conv_block(192, 384, kernel_size=3, padding=1),                      # 384 × 13 × 13
            conv_block(384, 256, kernel_size=3, padding=1),                      # 256 × 13 × 13
            conv_block(256, 256, kernel_size=3, padding=1, pool=True),           # 256 × 6 × 6
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


if __name__ == "__main__":
    model = AlexNet().eval()
    out = model(torch.randn(2, 3, 224, 224))
    assert out.shape == (2, 2), out.shape
    print("AlexNet ok", out.shape, "| tham số:", sum(p.numel() for p in model.parameters()))

In [ ]:
%%writefile train.py
"""Huấn luyện AlexNet từ đầu, theo khung train.py của repo ResNet-18, nhưng KHÔNG augmentation:
ảnh chỉ được cắt giữa 224×224 rồi chuẩn hóa theo mean/std của train.

Chạy: python train.py
Ghi ra: alexnet_scratch_best.pth (val accuracy cao nhất), alexnet_scratch_history.csv, alexnet_scratch.onnx
"""
import json

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from alexnet_scratch import AlexNet
from data_process import PROCESSED_FOLDER, STATS_FILE, IMAGE_SIZE, split_train_val


def load_data(batch_size=32):
    with open(STATS_FILE) as file:
        stats = json.load(file)

    # train, val, test dùng chung một phép biến đổi (ảnh trong Processed đã có cạnh ngắn 256)
    transform = transforms.Compose([
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(stats["mean"], stats["std"]),
    ])

    train_full = datasets.ImageFolder(PROCESSED_FOLDER + "/train", transform=transform)
    test_dataset = datasets.ImageFolder(PROCESSED_FOLDER + "/test", transform=transform)

    train_indices, val_indices = split_train_val(len(train_full))
    train_dataset = Subset(train_full, train_indices)
    val_dataset = Subset(train_full, val_indices)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=64, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=64, num_workers=4)

    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, loss_function, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.shape[0]
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += images.shape[0]

    return total_loss / total, correct / total


def evaluate(model, loader, loss_function, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = loss_function(outputs, labels)

            total_loss += loss.item() * images.shape[0]
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += images.shape[0]

    return total_loss / total, correct / total


def fit(model, name, loaders, lr, epochs, device="cuda" if torch.cuda.is_available() else "cpu"):
    train_loader, val_loader, test_loader = loaders
    model = model.to(device)
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0.0001)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_path = name + "_best.pth"
    best_val_accuracy = 0
    history = []

    for epoch in range(epochs):
        train_loss, train_accuracy = train_one_epoch(model, train_loader, loss_function, optimizer, device)
        val_loss, val_accuracy = evaluate(model, val_loader, loss_function, device)
        scheduler.step()
        history.append((epoch + 1, train_loss, train_accuracy, val_loss, val_accuracy))

        print(name, "epoch", epoch + 1, "|",
              "train loss", round(train_loss, 4), "acc", round(train_accuracy, 4), "|",
              "val loss", round(val_loss, 4), "acc", round(val_accuracy, 4))

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            torch.save(model.state_dict(), best_path)
            print("  saved best model")

    with open(name + "_history.csv", "w") as file:
        file.write("epoch,train_loss,train_accuracy,val_loss,val_accuracy\n")
        for row in history:
            file.write(",".join(str(value) for value in row) + "\n")

    model.load_state_dict(torch.load(best_path))
    test_loss, test_accuracy = evaluate(model, test_loader, loss_function, device)
    print(name, "test loss", round(test_loss, 4), "acc", round(test_accuracy, 4))

    model.eval()
    dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)  # batch phải > 1, nếu = 1 thì batch bị cố định
    try:  # cần gói onnxscript; nếu thiếu thì .pth và kết quả test ở trên vẫn còn nguyên
        torch.onnx.export(
            model, (dummy,), name + ".onnx",
            input_names=["image"], output_names=["logits"],
            dynamic_shapes={"x": {0: torch.export.Dim("batch", max=1024)}},
        )
    except Exception as error:
        print("bỏ qua xuất ONNX:", error)
    return test_accuracy


def train_scratch(epochs=30):
    torch.manual_seed(42)  # cố định khởi tạo trọng số và thứ tự xáo trộn ảnh
    return fit(AlexNet(num_classes=2), "alexnet_scratch", load_data(), lr=0.001, epochs=epochs)


if __name__ == "__main__":
    print("test accuracy:", train_scratch())

In [ ]:
# Huấn luyện: 30 epoch, lưu trọng số có val accuracy cao nhất, đánh giá test một lần
import time
import torch
import train
from alexnet_scratch import AlexNet
from data_process import PROCESSED_FOLDER

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có (chạy CPU)")
print("dữ liệu:", PROCESSED_FOLDER)
print("số tham số AlexNet:", sum(p.numel() for p in AlexNet().parameters()))

start = time.time()
test_accuracy = train.train_scratch()
print(f"thời gian huấn luyện: {(time.time() - start) / 60:.1f} phút")

In [ ]:
# Đường cong loss / accuracy (train đo trên chính ảnh đã học, không augmentation)
import pandas as pd
import matplotlib.pyplot as plt

history = pd.read_csv("alexnet_scratch_history.csv")
best_epoch = int(history.loc[history.val_accuracy.idxmax(), "epoch"])   # lần đầu đạt max, đúng epoch train.py đã lưu
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, metric in zip(axes, ["loss", "accuracy"]):
    axis.plot(history.epoch, history["train_" + metric], label="train")
    axis.plot(history.epoch, history["val_" + metric], label="val")
    axis.axvline(best_epoch, color="gray", linestyle="--", label=f"epoch tốt nhất ({best_epoch})")
    axis.set_xlabel("epoch")
    axis.set_title(metric)
    axis.legend()
plt.tight_layout()
plt.savefig("alexnet_scratch_curves.png", dpi=150)
plt.show()
print(history.round(4).to_string(index=False))

In [ ]:
# Báo cáo chi tiết trên test với trọng số tốt nhất + lưu dự đoán từng ảnh (cùng định dạng v1/v2 để so sánh McNemar)
import os
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AlexNet().to(device)
model.load_state_dict(torch.load("alexnet_scratch_best.pth", map_location=device))
model.eval()
_, _, test_loader = train.load_data()
test_dataset = test_loader.dataset                    # ImageFolder: 0 = chihuahua, 1 = muffin

probabilities = []
with torch.no_grad():
    for images, _ in test_loader:
        probabilities.append(model(images.to(device)).softmax(dim=1).cpu())
probabilities = torch.cat(probabilities).numpy()
labels = np.array(test_dataset.targets)
predictions = probabilities.argmax(axis=1)

print(classification_report(labels, predictions, target_names=test_dataset.classes, digits=4))
print("ROC-AUC (chihuahua là lớp dương):", round(roc_auc_score(labels == 0, probabilities[:, 0]), 4))
ConfusionMatrixDisplay(confusion_matrix(labels, predictions), display_labels=test_dataset.classes).plot(cmap="Blues")
plt.savefig("alexnet_scratch_cm.png", dpi=150)
plt.show()

pd.DataFrame({
    "key": [os.path.relpath(path, PROCESSED_FOLDER).replace("\\", "/") for path, _ in test_dataset.samples],
    "label": (labels == 0).astype(int),               # 1 = chihuahua, như test_predictions.csv của v1/v2
    "prob_chihuahua": probabilities[:, 0],
    "pred": (predictions == 0).astype(int),
}).to_csv("alexnet_scratch_test_predictions.csv", index=False)
print("đã lưu alexnet_scratch_test_predictions.csv")